In [1]:
import deepchem as dc
import pickle
import numpy as np

from sklearn.svm import SVR
from sklearn.metrics import r2_score

# Load ESOL
with open("../models/dataset_split.pkl", "rb") as f:
    train_dataset, valid_dataset, test_dataset, transformers = pickle.load(f)

param_grid = {
    "C": [0.1, 1, 10, 100],
    "gamma": ["scale", 0.01, 0.1, 1]
}

best_model = None
best_params = None
best_valid_r2 = -np.inf

No normalization for SPS. Feature removed!
No normalization for AvgIpc. Feature removed!
No normalization for NumAmideBonds. Feature removed!
No normalization for NumAtomStereoCenters. Feature removed!
No normalization for NumBridgeheadAtoms. Feature removed!
No normalization for NumHeterocycles. Feature removed!
No normalization for NumSpiroAtoms. Feature removed!
No normalization for NumUnspecifiedAtomStereoCenters. Feature removed!
No normalization for Phi. Feature removed!
Skipped loading material featurizer, missing a dependency. No module named 'ase'


Instructions for updating:
experimental_relax_shapes is deprecated, use reduce_retracing instead


Skipped loading modules with pytorch-geometric dependency, missing a dependency. No module named 'torch_geometric'
Skipped loading modules with transformers dependency. No module named 'transformers.models.roberta.tokenization_roberta_fast'
cannot import name 'Chemberta' from 'deepchem.models.torch_models' (/Users/sohanmulamalla/Desktop/Projects/soluscope-ai/venv/lib/python3.11/site-packages/deepchem/models/torch_models/__init__.py)
Skipped loading modules with pytorch-geometric dependency, missing a dependency. cannot import name 'DMPNN' from 'deepchem.models.torch_models' (/Users/sohanmulamalla/Desktop/Projects/soluscope-ai/venv/lib/python3.11/site-packages/deepchem/models/torch_models/__init__.py)
Skipped loading modules with pytorch-lightning dependency, missing a dependency. No module named 'lightning'
Skipped loading some Jax models, missing a dependency. No module named 'jax'


In [2]:
# Try each hyperparameter combination
for C in param_grid["C"]:
    for gamma in param_grid["gamma"]:

        model = SVR(
            kernel="rbf",
            C=C,
            gamma=gamma
        )

        model.fit(
            train_dataset.X,
            train_dataset.y.ravel()
        )

        valid_predictions = model.predict(valid_dataset.X)

        valid_r2 = r2_score(
            valid_dataset.y.ravel(),
            valid_predictions
        )

        if valid_r2 > best_valid_r2:
            best_valid_r2 = valid_r2
            best_params = {
                "C": C,
                "gamma": gamma,
                "kernel": "rbf"
            }
            best_model = model

print("Best params:", best_params)
print("Best validation R²:", best_valid_r2)

Best params: {'C': 100, 'gamma': 'scale', 'kernel': 'rbf'}
Best validation R²: 0.7461641412995587


In [3]:
test_predictions = best_model.predict(test_dataset.X)

y_test_actual = transformers[0].untransform(
    test_dataset.y
).flatten()

predictions_actual = transformers[0].untransform(
    test_predictions.reshape(-1, 1)
).flatten()

test_r2 = r2_score(
    y_test_actual,
    predictions_actual
)

print("Final Test R²:", test_r2)

Final Test R²: 0.7301767330798135


In [4]:
import pickle

with open("../models/best_solubility_model.pkl", "wb") as f:
    pickle.dump(best_model, f)

In [5]:
with open("../models/transformer.pkl", "wb") as f:
    pickle.dump(transformers, f)